# Build Metadata Map

In [ ]:
BEHAVIOR_LABEL_MAP2 = {
    "Fan": "SMALL_MOTOR_ELECTRONICS",
    "Vacuum": "SMALL_MOTOR_ELECTRONICS",
    "Washing Machine": "LAUNDRY_APPLIANCES",
    "Hairdryer": "THERMAL_APPLIANCES",
    "Fridge": "HVAC_REFRIGERATION",
    "Air Conditioner": "HVAC_REFRIGERATION",
    "Heater": "THERMAL_APPLIANCES",
    "Incandescent Light Bulb": "LIGHTING_LOADS",
    "Microwave": "THERMAL_APPLIANCES",
    "Laptop": "SMALL_MOTOR_ELECTRONICS",
    "Compact Fluorescent Lamp": "LIGHTING_LOADS"
}

In [ ]:
import json
from pathlib import Path

meta_path = Path("data/meta_2014.json")

with open(meta_path) as f:
    meta = json.load(f)

id_to_type = {}
for entry in meta:
    idx = int(entry["id"])
    typ = BEHAVIOR_LABEL_MAP2[entry["meta"]["appliance"]["type"]]
    id_to_type[idx] = typ

print("Unique appliance types:")
print(sorted(set(id_to_type.values())))


# Convert Every Recording to Power Series

In [ ]:
import numpy as np
import pandas as pd
from plaid_adapter import waveform_to_power_series

data_dir = Path("data/2014/2014")

power_series_by_type = {}

for idx, typ in id_to_type.items():

    file = data_dir / f"{idx}.csv"
    if not file.exists():
        print(f"[ERROR] File not found: {file}")
        continue

    df = pd.read_csv(file, header=None, names=['current', 'voltage'])

    voltage = df["voltage"].values
    current = df["current"].values

    power_series = waveform_to_power_series(voltage, current)

    power_series_by_type.setdefault(typ, []).append(power_series)


# Extract Per-Recording Statistics

In [ ]:
def extract_behavior_stats(power):
    power = np.array(power)

    steady = power[int(len(power)*0.6):]  # ignore transient part
    baseline = np.mean(steady)

    peak = np.max(power)
    spike_ratio = peak / baseline if baseline > 1 else 1

    # settling time: when signal enters steady band
    band = baseline * 0.1
    settling_idx = np.argmax(np.abs(power - baseline) < band)

    return {
        "mean_power": baseline,
        "std_steady": np.std(steady),
        "spike_ratio": spike_ratio,
        "settling_time": settling_idx,
        "variance": np.var(power)
    }


# Aggregate Per Appliance Type

In [ ]:
stats_by_type = {}

for typ, series_list in power_series_by_type.items():

    records = [extract_behavior_stats(s) for s in series_list]

    stats_by_type[typ] = pd.DataFrame(records)


In [ ]:
print(stats_by_type["SMALL_MOTOR_ELECTRONICS"].describe()[:3])

In [ ]:
classes = set(BEHAVIOR_LABEL_MAP2.values())

In [ ]:
for key in classes:
    print(key)
    print(stats_by_type[key].describe()[:3])
    print("\n")